# Idea 7 of the ideas planner: disjoint-half agreement (offline, from the stored matches)
For the candidate our pool chooses, re-fit the similarity from the LEFT half and from the RIGHT half of the canvas separately (matches split at the canvas middle; neighbouring headings reuse the same pixels, halves do not).
Rule fixed before looking (nothing tuned): H passes when both halves keep >= 3 inliers and the two half-fits put the UAV within 1 sigma of each other.
Questions: (a) does H separate right from wrong chosen candidates on both domains (AUC)? (b) as an OR-rescue of the photos ours skips: how many are rescued, how many wrong? (c) as a veto on ours: wrong answers removed vs right answers lost?
Sets: official150, nudt707, nudt1010 (seen only by the frozen decisions, no tuning here).

In [1]:
import os, sys, gc
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import cv2, math, numpy as np, pandas as pd
pd.set_option("display.width", 220)
from train.checks import replay
from fine_matching.loftr_matcher import stage2_from_matches
T7 = ROOT / "tests/Test_Iteration_7/results"
SETS = {"official150": "official150", "nudt707": "nudt_fresh707", "nudt1010": "nudt_fresh1010"}
MIN_HALF, CONF = 3, 0.22

def half_fit(q, rank, which, level, axis):
    """Fit of one half of the matches of candidate `rank` of query q. axis 0 = split by x (left/right), 1 = by y (top/bottom)."""
    m = q["matches"][rank]
    if m is None:
        return 0.0, np.zeros(2)
    mid = q["inside"].shape[axis] / 2.0
    keep = (m["points_a"][:, axis] < mid) if which == 0 else (m["points_a"][:, axis] >= mid)
    if keep.sum() < 4:
        return 0.0, np.zeros(2)
    item = dict(points_a=m["points_a"][keep], points_b=m["points_b"][keep], confidence=m["confidence"][keep])
    stored = dict(inside=q["inside"], margin=q["margin"], border_px=q["border_px"], items=[item])
    cv2.setRNGSeed(7)
    s2 = stage2_from_matches(stored, level, CONF)
    return float(s2["inliers"][0]), s2["shifts"][0]

def halves_of_choice(photo, ids, axis):
    cand = replay.pooled(photo, ids)
    k = replay.choose(cand)
    if k < 0:
        return None
    owner, rank = int(cand["owner"][k]), int(cand["rank"][k])
    q = photo["queries"][owner]
    px, py = q["peaks"][rank, 1], q["peaks"][rank, 2]
    n1, s1 = half_fit(q, rank, 0, photo["level"], axis)
    n2, s2 = half_fit(q, rank, 1, photo["level"], axis)
    d = math.hypot((s1[0] - s2[0]), (s1[1] - s2[1]))
    full_err = replay.error_sigma(photo, cand["x"][k], cand["y"][k])
    return dict(n_full=float(cand["inliers"][k]), n1=n1, n2=n2, half_dist_m=d, half_dist_sigma=d / photo["sigma"], err_sigma=full_err)

rows = []
for data, folder in SETS.items():
    ph = replay.load_set(T7 / "rich_dump" / folder, with_matches=True)
    for pid, photo in ph.items():
        r = replay.decide_current(photo)
        for pool, ids in [("std6", replay.query_ids(photo, replay.STANDARD)), ("wide27", replay.query_ids(photo, replay.WIDE))]:
            for axis, name in [(0, "LR"), (1, "TB")]:
                h = halves_of_choice(photo, ids, axis)
                if h is not None:
                    rows.append(dict(data=data, photo=pid, pool=pool, split=name, ours_stage=r["stage"], ours_pts=int(r["points"]), **h))
    del ph; gc.collect()
H = pd.DataFrame(rows)
H["pass"] = (np.minimum(H.n1, H.n2) >= MIN_HALF) & (H.half_dist_sigma <= 1.0)
H["right"] = H.err_sigma <= 4
H["wrong"] = H.err_sigma > 8
H.to_csv(T7 / "idea7_halves.csv", index=False)
print(len(H), "rows")

1700 rows


In [2]:
from sklearn.metrics import roc_auc_score
out = []
for (data, pool, split), g in H.groupby(["data", "pool", "split"]):
    lab = g[(g.right) | (g.wrong)]
    for feat, vals in [("min half inliers", np.minimum(lab.n1, lab.n2)), ("-half dist (sigma)", -lab.half_dist_sigma), ("pass flag", lab["pass"].astype(float))]:
        auc = roc_auc_score(lab.right, vals) if lab.right.nunique() == 2 else np.nan
        out.append(dict(data=data, pool=pool, split=split, feature=feat, n_right=int(lab.right.sum()), n_wrong=int(lab.wrong.sum()), auc=round(auc, 3)))
pd.DataFrame(out).pivot_table(index=["pool", "split", "feature"], columns="data", values="auc")

data                             nudt1010  nudt707  official150
pool   split feature                                           
std6   LR    -half dist (sigma)     0.782    0.742          NaN
             min half inliers       0.931    0.940          NaN
             pass flag              0.923    0.893          NaN
       TB    -half dist (sigma)     0.898    0.813          NaN
             min half inliers       0.931    0.931          NaN
             pass flag              0.923    0.927          NaN
wide27 LR    -half dist (sigma)     0.655    0.871        0.471
             min half inliers       0.948    0.929        0.869
             pass flag              0.940    0.843        0.848
       TB    -half dist (sigma)     0.672    0.846        0.928
             min half inliers       0.948    0.954        0.866
             pass flag              0.936    0.936        0.831

In [3]:
# (b) OR-rescue on photos ours skips; (c) veto on photos ours answers. Choice = std6 pool for photos ours answers at the bar/agreement, wide27 for skipped ones.
res = []
for data in SETS:
    for split in ["LR", "TB"]:
        a = H[(H.data == data) & (H.split == split)]
        skipped = a[(a.pool == "wide27") & (a.ours_stage == "skip")]
        resc = skipped[skipped["pass"]]
        gained = sum(replay.points(e) for e in resc.err_sigma)
        answered = a[(a.pool == "std6") & (a.ours_stage != "skip")]
        fail = answered[~answered["pass"]]
        res.append(dict(data=data, split=split, skipped=len(skipped), rescued=len(resc), rescued_right=int((resc.err_sigma <= 4).sum()), rescued_wrong=int((resc.err_sigma > 8).sum()),
                        gain_marks=round(gained / len(a[a.pool == "std6"]) * 6, 2), answered=len(answered), would_veto=len(fail), veto_wrong=int((fail.err_sigma > 8).sum()), veto_right=int((fail.err_sigma <= 4).sum())))
pd.DataFrame(res)

,data,split,skipped,rescued,rescued_right,rescued_wrong,gain_marks,answered,would_veto,veto_wrong,veto_right
0,official150,LR,14,0,0,0,0.00,133,40,0,40
1,official150,TB,14,0,0,0,0.00,133,41,0,41
2,nudt707,LR,12,1,0,1,-0.82,138,14,1,13
3,nudt707,TB,12,0,0,0,0.00,138,21,2,19
4,nudt1010,LR,7,0,0,0,0.00,124,19,1,18
5,nudt1010,TB,7,0,0,0,0.00,124,19,1,18
